# CROMA + lc-col BigEarthNet Phase 2A

Phase 2A uses CROMA optical-only on S2-only lc-col BigEarthNet data. This is not true SAR/optical sensor fairness.

Colab-first workflow for real Sentinel-2 chips from `lc-col/bigearthnet` with CROMA optical-only mode. Main path: 64-sample quick check, then 5000-sample Phase 2A run with chunked embeddings. No fake chips are generated.


In [ ]:
# 1. Clone or update this repo
# Edit this if your fork/repo URL differs.
REPO_URL = "https://github.com/strivekboy-coder/rsfm-fairness-audit.git"
REPO_DIR = "rsfm-fairness-audit"

from pathlib import Path
if Path(REPO_DIR).exists():
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}


In [ ]:
# 2. Install the package
!python -m pip install -e .


In [ ]:
# 3. Install CROMA + Hugging Face/HDF5 dependencies
!python -m pip install -r requirements-croma.txt


In [ ]:
# 4. Check GPU
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
# Configure CROMA for lc-col 12-band Sentinel-2 optical-only mode.
from pathlib import Path
import yaml

DATA_ROOT_64 = "data/bigearthnet_lccol_subset"
DATA_ROOT_5000 = "data/bigearthnet_lccol_subset5000"
OUTPUT_64 = "outputs/croma_bigearthnet_lccol64"
OUTPUT_5000 = "outputs/croma_bigearthnet_lccol5000"
CACHE_DIR = "data/_cache/lc_col_bigearthnet"
CROMA_REPO_DIR = "/content/CROMA"

if Path(CROMA_REPO_DIR).exists():
    !git -C {CROMA_REPO_DIR} pull
else:
    !git clone https://github.com/antofuller/CROMA {CROMA_REPO_DIR}

config_path = Path("configs/models/croma.yaml")
config = yaml.safe_load(config_path.read_text())
config["repo_path"] = CROMA_REPO_DIR
config["source_file_path"] = None
config["checkpoint_path"] = None
config["allow_hf_download"] = True
config["device"] = "auto"
config["input_modality"] = "optical"
config["expected_bands"] = 12
config["image_size"] = 120
config["embedding_key"] = "optical_GAP"
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config_path.read_text())


## Real lc-col BigEarthNet Data

The next cell downloads one real `lc-col/bigearthnet` HDF5 train shard from Hugging Face, inspects its HDF5 keys, and converts the first 64 real Sentinel-2 chips to this project's adapter format. The shard is large, so run this in Colab rather than committing data to Git.


In [ ]:
# Download one real lc-col/bigearthnet HDF5 shard and convert 64 real Sentinel-2 chips
!python scripts/download_bigearthnet_lccol_subset.py \
  --output-dir {DATA_ROOT_64} \
  --cache-dir {CACHE_DIR} \
  --max-samples 64 \
  --seed 42


In [ ]:
# Run preflight checker
!python -m rsfm_fairness_audit.cli check-real \
  --model croma \
  --dataset bigearthnet \
  --model-config configs/models/croma.yaml \
  --data-root {DATA_ROOT_64}


In [ ]:
# Run 64-sample real CROMA + lc-col BigEarthNet smoke test
!python -m rsfm_fairness_audit.cli run-real \
  --dataset bigearthnet \
  --dataset-root {DATA_ROOT_64} \
  --model croma \
  --config configs/models/croma.yaml \
  --output-dir {OUTPUT_64} \
  --max-samples 64


In [ ]:
# Inspect 64-sample outputs
!find {OUTPUT_64} -maxdepth 3 -type f -print
!sed -n '1,160p' {OUTPUT_64}/report.md

from IPython.display import Image, display
for fig in [f"{OUTPUT_64}/figures/average_vs_worst_group.png", f"{OUTPUT_64}/figures/fairness_map.png", f"{OUTPUT_64}/figures/raw_vs_balanced_gap.png"]:
    if Path(fig).exists():
        display(Image(filename=fig))


## 5000-Sample Phase 2A Run

Run this after the 64-sample quick check succeeds. It reuses `data/_cache/lc_col_bigearthnet` and enables chunked extraction to avoid Colab free-RAM OOM.


In [ ]:
# Convert 5000 real chips and run Phase 2A with chunked embeddings
!python scripts/download_bigearthnet_lccol_subset.py \
  --output-dir {DATA_ROOT_5000} \
  --cache-dir {CACHE_DIR} \
  --max-samples 5000 \
  --seed 42

!python -m rsfm_fairness_audit.cli run-real \
  --dataset bigearthnet \
  --dataset-root {DATA_ROOT_5000} \
  --model croma \
  --config configs/models/croma.yaml \
  --output-dir {OUTPUT_5000} \
  --max-samples 5000 \
  --chunk-size 256 \
  --streaming-embeddings true


In [ ]:
# Inspect 5000-sample Phase 2A outputs
!find {OUTPUT_5000} -maxdepth 3 -type f -print
!sed -n '1,160p' {OUTPUT_5000}/report.md

from IPython.display import Image, display
for fig in [f"{OUTPUT_5000}/figures/average_vs_worst_group.png", f"{OUTPUT_5000}/figures/fairness_map.png", f"{OUTPUT_5000}/figures/raw_vs_balanced_gap.png"]:
    if Path(fig).exists():
        display(Image(filename=fig))

!ls -lh {OUTPUT_5000}/tables


In [ ]:
# Package only the final selected Phase 2A run; exclude embeddings, chunks, predictions, data, and caches.
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

FINAL_RUN_DIR = Path(OUTPUT_5000)
ZIP_PATH = Path("croma_bigearthnet_phase2a_results.zip")
ROOT_ARTIFACTS = [
    "report.md",
    "fairness_summary.csv",
    "fairness_matrix_region.csv",
    "fairness_matrix_sensor.csv",
    "fairness_matrix_task.csv",
    "raw_vs_balanced_gap.csv",
    "classwise_metrics.csv",
    "probe_comparison.csv",
]

files_to_zip = []
for name in ROOT_ARTIFACTS:
    path = FINAL_RUN_DIR / name
    if path.exists():
        files_to_zip.append(path)
for folder_name in ["tables", "figures"]:
    folder = FINAL_RUN_DIR / folder_name
    if folder.exists():
        files_to_zip.extend(sorted(p for p in folder.rglob("*") if p.is_file()))

if not files_to_zip:
    raise RuntimeError(f"No report artifacts found in {FINAL_RUN_DIR}")

with ZipFile(ZIP_PATH, "w", compression=ZIP_DEFLATED) as archive:
    for path in files_to_zip:
        archive.write(path, path.relative_to(FINAL_RUN_DIR).as_posix())

print(f"Packaged {len(files_to_zip)} report artifacts from {FINAL_RUN_DIR} into {ZIP_PATH}")
print("Excluded: 64/512 outputs, embeddings.npz, predictions.csv, embedding_chunks/, data/, and HDF5 cache")
for path in files_to_zip:
    print(" -", path.relative_to(FINAL_RUN_DIR))

from google.colab import files
files.download(str(ZIP_PATH))


## Optional DOFA vs CROMA Comparison

Run this only if both final 5000-sample output folders already exist in the Colab workspace.


In [ ]:
# Optional comparison: run only when both DOFA and CROMA 5000 outputs exist.
from pathlib import Path

DOFA_5000 = Path("outputs/dofa_bigearthnet_lccol5000")
CROMA_5000 = Path("outputs/croma_bigearthnet_lccol5000")
COMPARISON_OUTPUT = "outputs/comparisons/dofa_vs_croma_lccol5000"

if DOFA_5000.exists() and CROMA_5000.exists():
    !python -m rsfm_fairness_audit.cli compare-runs \
      --dataset bigearthnet \
      --run dofa={DOFA_5000} \
      --run croma={CROMA_5000} \
      --output-dir {COMPARISON_OUTPUT}
    !find {COMPARISON_OUTPUT} -maxdepth 3 -type f -print
else:
    print("Skipping comparison because one or both 5000-sample output folders are missing.")


## Optional Cleanup

Uncomment these direct commands only after the final zip has downloaded successfully.


In [ ]:
# Optional cleanup: uncomment only after the final zip has downloaded successfully.
# !rm -rf data/bigearthnet_lccol_subset*
# !rm -rf data/_cache/lc_col_bigearthnet
# !rm -rf outputs/croma_bigearthnet_lccol64
# !rm -rf outputs/croma_bigearthnet_lccol5000/embedding_chunks
# !rm -f outputs/croma_bigearthnet_lccol5000/embeddings.npz
# !rm -f outputs/croma_bigearthnet_lccol5000/predictions.csv

# Optional: inspect remaining disk use after cleanup.
# !du -sh data outputs /root/.cache/huggingface /root/.cache/torch 2>/dev/null || true
